# Workshop — RAG sobre papers académicos con LangChain

En este workshop vas a construir un sistema RAG completo usando LangChain,
alimentado con dos papers fundamentales de IA:

- **attention_paper.pdf** — "Attention Is All You Need" (Vaswani et al., 2017)
- **rag_paper.pdf**       — "RAG: Retrieval-Augmented Generation" (Lewis et al., 2020)

Al final podrás hacerle preguntas a los papers y el sistema responderá
basándose únicamente en el contenido que hayas indexado.

**Stack:** MarkItDown · LangChain · Qdrant · Gemini 2.5 Flash Lite

**Estructura:**
| Parte | Tarea |
|---|---|
| 1 | Cargar y convertir PDFs con MarkItDown |
| 2 | Convertir a Documents de LangChain y chunkear |
| 3 | Crear embeddings e indexar en Qdrant |
| 4 | Construir el pipeline RAG con LCEL |
| 5 | Evaluar con las preguntas del quiz |

In [1]:
!pip install -q markitdown langchain langchain-google-genai langchain-qdrant
!pip install -q langchain-text-splitters qdrant-client python-dotenv

In [2]:
import os
from pathlib import Path
from typing import List
from dotenv import load_dotenv

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
assert GOOGLE_API_KEY, "Falta GOOGLE_API_KEY en .env"

print("Setup completo")

Setup completo


---
## Parte 1 — Cargar PDFs con MarkItDown

MarkItDown convierte cualquier PDF a texto Markdown con una sola línea.
El texto resultante preserva la estructura del documento (secciones, listas)
y es directamente usable como input para el chunker.

**Tu tarea:** completar la función `load_pdf()` y cargar los dos papers.

In [4]:
from markitdown import MarkItDown

md_converter = MarkItDown(enable_plugins=False)

DOCS_DIR = Path("papers")

def load_pdf(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    result = md_converter.convert(str(path))
    text = result.text_content
    if not text or not text.strip():
        raise ValueError(f"No se pudo extraer texto de {path.name}. "
                         "Verifica que el PDF no sea escaneado.")
    return text


attention_text = load_pdf(DOCS_DIR / "attention_paper.pdf")
rag_text       = load_pdf(DOCS_DIR / "rag_paper.pdf")

print(f"attention_paper.pdf → {len(attention_text):,} chars")
print(f"rag_paper.pdf       → {len(rag_text):,} chars")
print(f"\nPrimeros 200 chars del paper de Attention:\n{attention_text[:200]}...")

attention_paper.pdf → 39,662 chars
rag_paper.pdf       → 69,390 chars

Primeros 200 chars del paper de Attention:
3
2
0
2

g
u
A
2

]
L
C
.
s
c
[

7
v
2
6
7
3
0
.
6
0
7
1
:
v
i
X
r
a

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely ...


---
## Parte 2 — Convertir a Documents y chunkear

LangChain trabaja con objetos `Document` que tienen:
- `page_content` — el texto del chunk
- `metadata`     — dict con información del origen (título, fuente, etc.)

**Tu tarea:** completar `pdf_to_documents()` que convierte el texto de un PDF
en una lista de `Document` usando `RecursiveCharacterTextSplitter`.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# ── TODO 2 ───────────────────────────────────────────────────
# Completa la función pdf_to_documents():
# - Crea un RecursiveCharacterTextSplitter con chunk_size=600, chunk_overlap=60
# - Usa text_splitter.split_text(text) para obtener los chunks
# - Por cada chunk crea un Document con:
#     page_content = chunk
#     metadata = {"title": title, "source": source, "chunk_idx": i}
# - Retorna la lista de Documents
# - Descarta chunks con menos de 30 caracteres

def pdf_to_documents(text: str, title: str, source: str) -> List[Document]:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=60,
        separators=["\n\n", "\n", ".", " "],
    )
    chunks = text_splitter.split_text(text)
    docs = []
    for i, chunk in enumerate(chunks):
        if len(chunk.strip()) < 30:
            continue
        docs.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_idx": i},
        ))
    return docs


# ── Aplicar a los dos papers ──────────────────────────────────
attention_docs = pdf_to_documents(
    attention_text,
    title="Attention Is All You Need",
    source="attention_paper.pdf",
)
rag_docs = pdf_to_documents(
    rag_text,
    title="Retrieval-Augmented Generation",
    source="rag_paper.pdf",
)

all_docs = attention_docs + rag_docs

print(f"attention_paper → {len(attention_docs)} chunks")
print(f"rag_paper       → {len(rag_docs)} chunks")
print(f"Total           → {len(all_docs)} chunks")
print(f"\nEjemplo de Document:")
print(f"  page_content: {all_docs[0].page_content[:100]}...")
print(f"  metadata:     {all_docs[0].metadata}")

attention_paper → 88 chunks
rag_paper       → 166 chunks
Total           → 254 chunks

Ejemplo de Document:
  page_content: 3
2
0
2

g
u
A
2

]
L
C
.
s
c
[

7
v
2
6
7
3
0
.
6
0
7
1
:
v
i
X
r
a

Provided proper attribution is...
  metadata:     {'title': 'Attention Is All You Need', 'source': 'attention_paper.pdf', 'chunk_idx': 0}


---
## Parte 3 — Embeddings e índice Qdrant

Ahora indexamos los Documents en Qdrant usando Gemini para los embeddings.

**Tu tarea:** inicializar el `GoogleGenerativeAIEmbeddings` y crear el
`QdrantVectorStore` indexando todos los documentos.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "papers_rag"
EMBEDDING_DIM = 768   # gemini-embedding-001 con output_dimensionality=768

# ── TODO 3a ──────────────────────────────────────────────────
# Inicializa GoogleGenerativeAIEmbeddings con:
# - model="models/gemini-embedding-001"
# - google_api_key=GOOGLE_API_KEY
# - output_dimensionality=768

embeddings = None  # TODO: reemplazar con la inicialización correcta

# ── TODO 3b ──────────────────────────────────────────────────
# Crea un QdrantClient en memoria y la colección "papers_rag"
# con Distance.COSINE y size=EMBEDDING_DIM

qdrant_client = None  # TODO

# ── TODO 3c ──────────────────────────────────────────────────
# Crea un QdrantVectorStore con el cliente, la colección y los embeddings
# Luego indexa all_docs con vectorstore.add_documents(all_docs)

vectorstore = None  # TODO

print(f"Indexados {len(all_docs)} chunks en Qdrant")

---
## Parte 4 — Pipeline RAG con LCEL

Con el índice listo, construimos el pipeline RAG usando el operador `|` de LangChain.

**Tu tarea:** construir el pipeline completo conectando retriever → prompt → LLM.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── TODO 4a ──────────────────────────────────────────────────
# Crea el LLM con ChatGoogleGenerativeAI:
# - model="gemini-2.5-flash-lite"  (el más rápido del free tier)
# - temperature=0.1

llm = None  # TODO

# ── TODO 4b ──────────────────────────────────────────────────
# Crea el retriever a partir del vectorstore:
# - search_type="similarity_score_threshold"
# - k=4, score_threshold=0.3

retriever = None  # TODO

# ── TODO 4c ──────────────────────────────────────────────────
# Implementa format_docs() que convierte List[Document] a string de contexto
# Formato sugerido: "[Título]\ncontenido\n\n---\n\n[Título]\ncontenido..."

def format_docs(docs: List[Document]) -> str:
    """Convierte lista de Documents a string de contexto para el prompt."""
    # TODO
    raise NotImplementedError("Implementa format_docs()")

# ── TODO 4d ──────────────────────────────────────────────────
# Crea el prompt template con las variables {context} y {question}
# El sistema debe responder SOLO basándose en el contexto proporcionado

prompt = None  # TODO: ChatPromptTemplate.from_template(...)

# ── TODO 4e ──────────────────────────────────────────────────
# Construye el pipeline LCEL:
# rag_chain = (
#     {"context": retriever | format_docs, "question": RunnablePassthrough()}
#     | prompt | llm | StrOutputParser()
# )

rag_chain = None  # TODO

# ── Prueba rápida ─────────────────────────────────────────────
test_answer = rag_chain.invoke("What is the Transformer architecture?")
print(f"Prueba:\n{test_answer[:300]}...")

---
## Parte 5 — Quiz: preguntas sobre los papers

Responde estas preguntas con tu sistema RAG.
Las respuestas correctas están al final — compara con lo que genera el sistema.

Primero corre cada celda y observa la respuesta del RAG.
Luego verifica contra el ground truth.

In [ ]:
# ── Pregunta 1 ────────────────────────────────────────────────
q1 = "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"

answer1 = rag_chain.invoke(q1)
print(f"Q: {q1}")
print(f"A: {answer1}")

<details>
<summary>✅ Ground Truth — Pregunta 1</summary>

El Transformer (big model) obtuvo un BLEU score de **28.4** en la tarea de
traducción WMT 2014 English-to-German, superando los mejores resultados
previos (incluyendo ensembles) por más de 2 puntos BLEU.
*(Fuente: Abstract y Section 6.1 — Attention Is All You Need)*
</details>

In [ ]:
# ── Pregunta 2 ────────────────────────────────────────────────
q2 = "How does RAG combine parametric and non-parametric memory?"

answer2 = rag_chain.invoke(q2)
print(f"Q: {q2}")
print(f"A: {answer2}")

<details>
<summary>✅ Ground Truth — Pregunta 2</summary>

RAG combina **memoria paramétrica** (un modelo seq2seq pre-entrenado, BART-large)
con **memoria no-paramétrica** (un índice denso de vectores de Wikipedia).
El retriever (DPR) recupera los top-K documentos relevantes para la query,
y el generador (BART) los usa como contexto adicional para producir la respuesta.
Ambos componentes se entrenan end-to-end de forma conjunta.
*(Fuente: Abstract y Section 2 — RAG paper)*
</details>

In [ ]:
# ── Pregunta 3 ────────────────────────────────────────────────
q3 = "What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?"

answer3 = rag_chain.invoke(q3)
print(f"Q: {q3}")
print(f"A: {answer3}")

<details>
<summary>✅ Ground Truth — Pregunta 3</summary>

El factor de escala **1/√dk** se aplica para contrarrestar el efecto de que,
con valores grandes de dk, los productos punto crecen en magnitud y empujan
la función softmax hacia regiones con gradientes muy pequeños.
Esto haría el entrenamiento inestable. La división por √dk mantiene los
productos punto en un rango con gradientes útiles.
*(Fuente: Section 3.2.1 — Attention Is All You Need)*
</details>

In [ ]:
# ── Pregunta 4 ────────────────────────────────────────────────
q4 = "What are the two RAG model variants proposed in the paper and how do they differ?"

answer4 = rag_chain.invoke(q4)
print(f"Q: {q4}")
print(f"A: {answer4}")

<details>
<summary>✅ Ground Truth — Pregunta 4</summary>

Los dos variantes son:
- **RAG-Sequence**: usa el **mismo documento recuperado** para generar toda
  la secuencia de salida. Trata el documento como una variable latente única.
- **RAG-Token**: puede usar un **documento diferente para cada token** generado,
  permitiendo combinar información de múltiples fuentes en una sola respuesta.

RAG-Token es más flexible pero más costoso computacionalmente.
*(Fuente: Section 2.1 — RAG paper)*
</details>

In [ ]:
# ── Pregunta 5 ────────────────────────────────────────────────
q5 = "How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?"

answer5 = rag_chain.invoke(q5)
print(f"Q: {q5}")
print(f"A: {answer5}")

<details>
<summary>✅ Ground Truth — Pregunta 5</summary>

El Transformer base usa **h = 8** attention heads en paralelo.
Para cada cabeza, las dimensiones son **dk = dv = dmodel/h = 512/8 = 64**.
El costo computacional total es similar al de single-head attention con
dimensionalidad completa, gracias a las dimensiones reducidas por cabeza.
*(Fuente: Section 3.2.2 — Attention Is All You Need)*
</details>

---
## Bonus — Pregunta cross-paper

Esta pregunta requiere información de **ambos** papers a la vez.
Observa si el RAG recupera chunks de los dos documentos.

In [ ]:
q_bonus = ("The RAG paper uses FAISS for fast retrieval with an HNSW approximation. "
           "The Attention paper proposes the Transformer architecture. "
           "What seq2seq model does RAG use as its generator component, "
           "and is it based on the Transformer?")

answer_bonus = rag_chain.invoke(q_bonus)

# Mostrar también qué chunks recuperó
docs_retrieved = retriever.invoke(q_bonus)
sources = [(d.metadata["title"], d.metadata["source"]) for d in docs_retrieved]

print(f"Q: {q_bonus}\n")
print(f"A: {answer_bonus}\n")
print(f"Fuentes recuperadas:")
for title, source in sources:
    print(f"  [{title}] — {source}")

<details>
<summary>✅ Ground Truth — Bonus</summary>

RAG usa **BART-large** como generador (Section 2.3 del RAG paper).
BART es un modelo seq2seq pre-entrenado basado en la arquitectura **Transformer**
(referencia [58] en el RAG paper = el paper "Attention Is All You Need").
El RAG paper cita explícitamente: "BART-large, a pre-trained seq2seq transformer".

Esta pregunta demuestra el valor de RAG: el sistema conecta información de
dos documentos distintos para construir una respuesta completa.
</details>

---
## Reflexión final

Responde estas preguntas basándote en lo que observaste durante el workshop:

1. **¿Qué pasó cuando el RAG no encontró información relevante?**
   ¿Respondió correctamente que no sabía, o inventó algo?

2. **¿Qué chunks recuperó para la pregunta bonus?**
   ¿Fueron de ambos papers o solo de uno?

3. **¿Qué cambiarías del pipeline** para mejorar la calidad de las respuestas?
   Piensa en: chunk_size, score_threshold, k, el prompt.